<a href="https://colab.research.google.com/github/NandakrishnanR/Kaggle_projects/blob/master/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xgboost
!pip install opendatasets

In [ ]:
import os
import opendatasets as od
import pandas as pd
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
od.download('https://www.kaggle.com/c/rossmann-store-sales')

In [ ]:
import pandas as pd
train_df = pd.read_csv('/content/rossmann-store-sales/train.csv', low_memory=False)
display(train_df.head())
test_df = pd.read_csv('/content/rossmann-store-sales/test.csv', low_memory=False)
store_df = pd.read_csv('/content/rossmann-store-sales/store.csv', low_memory=False)
sample_df = pd.read_csv('/content/rossmann-store-sales/sample_submission.csv', low_memory=False)



# Feature Engineering


1.   here initialy i merge both tables
2.  find relation where sales=0 has any impact



1.   ignore customer column as it is not in test data
2.   Columns date,compeition and promo have more things to say so either we can merge or find a new relationship for them and create new column













In [ ]:
merged_df= pd.merge(train_df, store_df, on='Store')
merged_df.isna().sum()
merged_test_df=pd.merge(test_df,store_df,on='Store')

In [ ]:
merged_df.corr(numeric_only=True)

In [ ]:
merged_df[merged_df["StateHoliday"]!="0"]
(merged_df["Sales"]==0).sum()

In [ ]:
zero_sales_df = merged_df[merged_df['Sales'] == 0]
print("Shape of zero_sales_df:", zero_sales_df.shape)
display(zero_sales_df.head())

In [ ]:
merged_df = merged_df.drop('Customers', axis=1)
merged_df['Date'] = pd.to_datetime(merged_df['Date'])
merged_df['Year'] = merged_df['Date'].dt.year
merged_df['Month'] = merged_df['Date'].dt.month
merged_df['Day'] = merged_df['Date'].dt.day
merged_df['WeekOfYear'] = merged_df['Date'].dt.isocalendar().week.astype(int)
display(merged_df.head())

In [ ]:
def split_date(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df.Date.dt.year
    df['Month'] = df.Date.dt.month
    df['Day'] = df.Date.dt.day
    df['WeekOfYear'] = df.Date.dt.isocalendar().week

split_date(merged_df)
display(merged_df.head())
split_date(merged_test_df)
display(merged_test_df.head())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

categorical_features = [
    'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday',
    'StoreType', 'Assortment', 'Promo2', 'PromoInterval', 'Year', 'Month'
]

# Determine grid dimensions
num_features = len(categorical_features)
num_cols = 4  # Example: 4 columns per row
num_rows = (num_features + num_cols - 1) // num_cols # Calculate rows needed

plt.figure(figsize=(num_cols * 5, num_rows * 4))

# Create a copy for plotting to handle NaN values without modifying the original DataFrame
plot_data = merged_df.copy()
plot_data['PromoInterval'] = plot_data['PromoInterval'].fillna('No Promo')
plot_data['StateHoliday'] = plot_data['StateHoliday'].replace({'0': 'No Holiday', 'a': 'Public Holiday', 'b': 'Easter Holiday', 'c': 'Christmas'})

for i, feature in enumerate(categorical_features):
    ax = plt.subplot(num_rows, num_cols, i + 1)
    sns.barplot(data=plot_data, x=feature, y='Sales', ax=ax, palette='viridis', hue=feature, legend=False)
    ax.set_title(f'Average Sales by {feature}')
    ax.set_xlabel(feature)
    ax.set_ylabel('Average Sales')
    if len(plot_data[feature].unique()) > 5: # Rotate labels if too many categories
        ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numerical_features = [
    'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
    'Promo2SinceWeek', 'Promo2SinceYear'
]

# Determine grid dimensions
num_features = len(numerical_features)
num_cols = 3  # Example: 3 columns per row
num_rows = (num_features + num_cols - 1) // num_cols # Calculate rows needed

plt.figure(figsize=(num_cols * 6, num_rows * 5))

for i, feature in enumerate(numerical_features):
    ax = plt.subplot(num_rows, num_cols, i + 1)
    # Drop NaN values for the current feature and 'Sales' to ensure clean plots
    plot_data_filtered = merged_df[[feature, 'Sales']].dropna()
    sns.scatterplot(data=plot_data_filtered, x=feature, y='Sales', ax=ax, alpha=0.5)
    ax.set_title(f'Sales vs. {feature}')
    ax.set_xlabel(feature)
    ax.set_ylabel('Sales')

plt.tight_layout()
plt.show()

*#compeition column has more meanings so i extracted them to months calculated form respective date*

In [ ]:
def comp_months(df):
    # Calculate raw months difference
    df['CompetitionOpenMonths'] = 12 * (df.Year - df.CompetitionOpenSinceYear) + \
                                  (df.Month - df.CompetitionOpenSinceMonth)

    # Handle cases where competition started after the observation date (negative values)
    # and also fill NaNs for stores with no competition info, by setting them to 0.
    # Using .where() to selectively replace negative values, then fillna.
    df['CompetitionOpenMonths'] = df['CompetitionOpenMonths'].where(df['CompetitionOpenMonths'] >= 0, 0).fillna(0)

# Apply the function to merged_df
comp_months(merged_df)
comp_months(merged_test_df)

print("First 5 rows with new 'CompetitionOpenMonths' column:")
display(merged_df[['Date', 'CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'CompetitionOpenMonths']].head())

print("\nNumber of missing values in 'CompetitionOpenMonths':")
print(merged_df['CompetitionOpenMonths'].isnull().sum())

In [ ]:
#promo2sinceyear and promo2sinceweek can be extracted to a single feature
#here i checked if promo2 is 0,other columnsa re also 0
promo2_is_zero_df = merged_df[merged_df['Promo2'] == 0]

print(f"Number of rows where Promo2 is 0: {len(promo2_is_zero_df)}")
print("\nMissing values in 'Promo2SinceYear' when Promo2 is 0:")
print(promo2_is_zero_df['Promo2SinceYear'].isnull().sum())
print("\nMissing values in 'Promo2SinceWeek' when Promo2 is 0:")
print(promo2_is_zero_df['Promo2SinceWeek'].isnull().sum())

In [ ]:
def calculate_promo2_duration(df):
    # Initialize Promo2DurationMonths to 0 for all rows
    df['Promo2DurationMonths'] = 0

    # Identify rows with active Promo2 participation
    active_promo2_mask = (df['Promo2'] == 1)

    # For active Promo2 rows, construct Promo2StartDate
    # Combine Promo2SinceYear and Promo2SinceWeek into a 'YYYY-WW-1' string
    promo2_start_date_str = df.loc[active_promo2_mask].apply(lambda row: f"{int(row['Promo2SinceYear'])}-{int(row['Promo2SinceWeek'])}-1", axis=1)
    promo2_start_date = pd.to_datetime(promo2_start_date_str, format='%Y-%W-%w')

    # Calculate duration in months for active Promo2 rows
    duration_months = (df.loc[active_promo2_mask, 'Date'].dt.year - promo2_start_date.dt.year) * 12 + \
                      (df.loc[active_promo2_mask, 'Date'].dt.month - promo2_start_date.dt.month)

    # Update Promo2DurationMonths in the DataFrame
    df.loc[active_promo2_mask, 'Promo2DurationMonths'] = duration_months.astype(int)

    # Ensure no negative values are present
    df['Promo2DurationMonths'] = df['Promo2DurationMonths'].apply(lambda x: max(0, x))

# Apply the function to merged_df and merged_test_df
calculate_promo2_duration(merged_df)
calculate_promo2_duration(merged_test_df)

print("First 5 rows of merged_df with 'Promo2DurationMonths' column:")
display(merged_df[['Date', 'Promo2', 'Promo2SinceYear', 'Promo2SinceWeek', 'Promo2DurationMonths']].head())

print("First 5 rows of merged_test_df with 'Promo2DurationMonths' column:")
display(merged_test_df[['Date', 'Promo2', 'Promo2SinceYear', 'Promo2SinceWeek', 'Promo2DurationMonths']].head())

# Input and Target columns

In [ ]:
merged_df.columns

In [ ]:
input_col=['Store','DayOfWeek','Open','Promo','StateHoliday','SchoolHoliday','StoreType','Assortment','CompetitionDistance','CompetitionOpenMonths','Promo2','Promo2DurationMonths', 'PromoInterval', 'Year', 'Month','Day', 'WeekOfYear']
target_col=['Sales']

In [ ]:
train_input=merged_df[input_col]
train_target=merged_df[target_col]
test_input=merged_test_df[input_col]

# Numerical and categorical

In [ ]:
numerical_col = ['Store','DayOfWeek','Open','Promo','SchoolHoliday','CompetitionDistance','CompetitionOpenMonths','Promo2','Promo2DurationMonths','Year','Month','Day','WeekOfYear']
categorical_col = ['StateHoliday','StoreType','Assortment', 'PromoInterval']
train_input[numerical_col].isna().sum()
train_input[categorical_col].isna().sum()

# Imputing

In [ ]:
#here if we use anyother imputer it will change the meaning,so i am putting max distance which means less impact or similar to putting a real zero
max_value=train_input.CompetitionDistance.max()
train_input.CompetitionDistance.fillna(max_value,inplace=True)
test_input.CompetitionDistance.fillna(max_value,inplace=True)

In [ ]:
#here cateogircal is column so later in onehot ncoding it splits into 2 meanginfull columns
train_input.PromoInterval = train_input['PromoInterval'].fillna('No Promo')
test_input.PromoInterval = test_input['PromoInterval'].fillna('No Promo')

**Scaling**

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(train_input[numerical_col])
train_input[numerical_col]=scaler.transform(train_input[numerical_col])
test_input[numerical_col]=scaler.transform(test_input[numerical_col])

In [ ]:
#TO KNOW IF THE CATEGORICAL COLUMN IS CATEGORY ITSELF OR STRING VALUES
train_input[categorical_col].nunique()

# One hot enoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
encoder.fit(train_input[categorical_col])
encoded_col=encoder.get_feature_names_out(categorical_col)
train_input[encoded_col]=encoder.transform(train_input[categorical_col])
test_input[encoded_col]=encoder.transform(test_input[categorical_col])
train_input[encoded_col]

In [ ]:
final_features = numerical_col + encoded_col.tolist()

# Select these columns from the preprocessed DataFrames
X = train_input[final_features]
X_test = test_input[final_features]

# Training

In [ ]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(a, b):
    # In newer scikit-learn versions, mean_squared_error always returns MSE.
    # To get RMSE, we calculate MSE and then take its square root.
    return np.sqrt(mean_squared_error(a, b))

kfold = KFold(n_splits=5)

models = []
fold_rmses = [] # Initialize fold_rmses

for train_idxs, val_idxs in kfold.split(X):
    X_train_fold, train_target_fold = X.iloc[train_idxs], train_target.iloc[train_idxs]
    X_val_fold, val_target_fold = X.iloc[val_idxs], train_target.iloc[val_idxs]

    model = XGBRegressor(
        random_state=42,
        n_jobs=-1,
        max_depth=4,
        n_estimators=20
    )
    model.fit(X_train_fold, train_target_fold)

    train_rmse = rmse(model.predict(X_train_fold), train_target_fold)
    val_rmse = rmse(model.predict(X_val_fold), val_target_fold)

    models.append(model)
    fold_rmses.append(val_rmse) # Append validation RMSE to fold_rmses
    print('Train RMSE: {}, Validation RMSE: {}'.format(train_rmse, val_rmse))

print(f"\nAverage K-Fold RMSE: {np.mean(fold_rmses):.4f}")
# The standard deviation of the RMSEs across folds indicates how consistent the model's performance is.
# A lower standard deviation suggests a more stable and reliable model across different subsets of the data.
print(f"Standard Deviation of K-Fold RMSEs: {np.std(fold_rmses):.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#visualization
import matplotlib.pyplot as plt
from xgboost import plot_tree
from matplotlib.pylab import rcParams
%matplotlib inline

rcParams['figure.figsize'] = 30,30
plot_tree(model, rankdir='LR');

# Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
import seaborn as sns
plt.figure(figsize=(10,6))
plt.title('Feature Importance')
sns.barplot(data=importance_df.head(10), x='importance', y='feature');

# Final predicition and submission

In [ ]:
submission_df = pd.read_csv('/content/rossmann-store-sales/sample_submission.csv')
display(submission_df.head())

In [ ]:
test_preds = model.predict(X_test)

submission_df['Sales']  = test_preds
test_df.Open.isna().sum()
submission_df['Sales'] = submission_df['Sales'] * test_df.Open.fillna(1.)

In [ ]:
submission_df

In [ ]:
(submission_df['Sales'] == 0).sum()

In [ ]:
submission_df.to_csv('submission.csv', index=False)


In [ ]:
#try
test1={
    "Id":"4",
    "Store": "2",
    "DayOfWeek":"4",
    "Date":"2016-09-17"	,
    "Open":0,
    "Promo":1,
    "StateHoliday":"a",
    "SchoolHoliday":0}
test_df=pd.DataFrame([test1])


In [ ]:
test_df

In [ ]:
split_date(test_df)
input_merge = test_df.merge(store_df, on='Store', how='left')
calculate_promo2_duration(input_merge)
comp_months(input_merge)
input_merge

In [ ]:
final_test=input_merge[input_col].copy()
final_test

In [ ]:
final_test[numerical_col]=scaler.transform(final_test[numerical_col])

In [114]:
final_test[encoded_col]=encoder.transform(final_test[categorical_col])
X_test2=final_test[final_features]
model.predict(X_test2)[0]

np.float32(-474.3712)